# 11.1 — The Flat Key Space

**Chapter 11, §2.2 "The Flat Namespace of Object Storage" (and Exercise 2).**

The chapter makes three claims about what changes when a hierarchical file system is replaced by
a bucket of keys:

1. there is no atomic rename — moving a prefix is $N$ copies and $N$ deletes;
2. there are no directories, so listing a prefix is a scan of the key index;
3. there is no multi-object transaction, so a job that fails half way leaves a half-written table.

This notebook makes all three visible. It uses **no Spark**: the point is a property of the
storage layer, and a Spark session would only hide it behind a commit protocol. The local
filesystem plays the part of the POSIX side, and a dictionary of keys plays the part of the
bucket.

In [1]:
import os, shutil, tempfile

# Everything happens in scratch. Nothing in this notebook touches the repository.
SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch11-keyspace")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH)
SCRATCH

'/var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/ch11-keyspace'

## 1. The POSIX side: rename is one call

A small table laid out the way the chapter's left-hand panel draws it — `sales/` with two months
under one year, and three data files in all.

In [2]:
FS = os.path.join(SCRATCH, "fs")
paths = [
    "sales/year=2025/month=06/part-000.parquet",
    "sales/year=2025/month=07/part-000.parquet",
    "sales/year=2025/month=07/part-001.parquet",
]
for p in paths:
    full = os.path.join(FS, p)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "wb") as f:
        f.write(b"x" * 1024)

found = [os.path.relpath(os.path.join(root, name), FS)
         for root, dirs, files in os.walk(FS) for name in files]
print("\n".join(sorted(found)))

sales/year=2025/month=06/part-000.parquet
sales/year=2025/month=07/part-000.parquet
sales/year=2025/month=07/part-001.parquet


In [3]:
# Publishing the table means moving sales/ to published/. On a file system that is one
# syscall, and it does not touch a single byte of data.
os.rename(os.path.join(FS, "sales"), os.path.join(FS, "published"))
print("calls made:", 1)
print(sorted(os.listdir(FS)))

calls made: 1
['published']


The directory entry is a pointer. Renaming it rewrites the pointer; the three files never move,
and there is no instant at which some of them are published and some are not.

## 2. The object-store side: there is no rename

A bucket is a map from a key to bytes. It offers `put`, `get`, `list(prefix)`, `copy` and
`delete` — and no rename, because there is nothing to rename: the slashes in the key are
characters, not structure.

In [4]:
class Bucket:
    # A flat key space. Every method that reaches the service counts as one request.

    def __init__(self):
        self._keys = {}
        self.requests = 0

    def put(self, key, data):
        self.requests += 1
        self._keys[key] = data

    def get(self, key):
        self.requests += 1
        return self._keys[key]

    def delete(self, key):
        self.requests += 1
        del self._keys[key]

    def copy(self, src, dst):
        self.requests += 1
        self._keys[dst] = self._keys[src]

    def list(self, prefix, page_size=1000):
        # No directories: the service scans its key index in lexicographic order and
        # returns one page of matching keys per request.
        keys = sorted(k for k in self._keys if k.startswith(prefix))
        pages = max(1, -(-len(keys) // page_size))
        self.requests += pages
        return keys

In [5]:
bucket = Bucket()
for p in paths:
    bucket.put(p, b"x" * 1024)
bucket.requests = 0

# "Renaming" sales/ to published/ : every key must be copied to a new key and the old one
# deleted. There is no single call that does it.
for key in bucket.list("sales/"):
    bucket.copy(key, key.replace("sales/", "published/", 1))
    bucket.delete(key)

print("requests for 3 objects:", bucket.requests)
print(bucket.list("published/"))

requests for 3 objects: 7
['published/year=2025/month=06/part-000.parquet', 'published/year=2025/month=07/part-000.parquet', 'published/year=2025/month=07/part-001.parquet']


One list, then a copy and a delete per object. The cost is in the number of objects, not in the
number of path components — so it grows with the table.

In [6]:
def rename_prefix(n):
    b = Bucket()
    for i in range(n):
        b.put(f"sales/part-{i:05d}.parquet", b"")
    b.requests = 0
    for key in b.list("sales/"):
        b.copy(key, key.replace("sales/", "published/", 1))
        b.delete(key)
    return b.requests

print(f"{'objects':>8}  {'requests':>9}")
for n in (10, 100, 1000):
    print(f"{n:>8}  {rename_prefix(n):>9}")

 objects   requests
      10         21
     100        201
    1000       2001


$2N + 1$. The chapter's $O(N)$, and each of those requests is an HTTP round trip rather than a
pointer write. This is why the classical Spark commit protocol — stage the output in a temporary
directory, rename it into place when the job finishes — does not transfer to object storage:
the rename it depends on is neither cheap nor atomic.

## 3. Listing: the cost is in the keys, not the depth

A POSIX listing of one directory reads one directory. A prefix listing reads the bucket's key
index, and a table split into many small files pays for every one of them during query planning.

In [7]:
b = Bucket()
for i in range(2500):
    b.put(f"events/part-{i:05d}.parquet", b"")
b.requests = 0
found = b.list("events/", page_size=1000)
print(f"{len(found)} objects discovered in {b.requests} requests "
      f"({b.requests} paginated round trips before any data is read)")

2500 objects discovered in 3 requests (3 paginated round trips before any data is read)


## 4. No multi-object transaction

The chapter's third constraint: "If a distributed batch job fails after writing 50 out of 100
output objects, the target prefix is left in an inconsistent, partially updated state."

Each `put` is atomic on its own. Nothing makes a hundred of them atomic together.

In [8]:
class Failure(Exception):
    pass


def write_table(bucket, prefix, n_objects, fail_after=None):
    for i in range(n_objects):
        if fail_after is not None and i == fail_after:
            raise Failure(f"executor lost after {i} objects")
        bucket.put(f"{prefix}part-{i:05d}.parquet", b"row")


b = Bucket()
try:
    write_table(b, "orders/", 100, fail_after=50)
except Failure as exc:
    print("job failed:", exc)

# A reader arriving now sees a table. It is simply the wrong table.
print("objects a reader would find under orders/ :", len(b.list("orders/")))

job failed: executor lost after 50 objects
objects a reader would find under orders/ : 50


Nothing is corrupt and nothing is incomplete *as an object*; all fifty are whole. The damage is
that the set is wrong, and the storage layer has no opinion about sets.

## 5. What fixes it: one key that says which objects count

The repair is not to make the writes atomic. It is to stop defining the table as "the objects
under this prefix" and define it as "the objects named by this one small key" — which can then be
replaced in a single atomic `put`.

In [9]:
import json

b = Bucket()

# The failed job wrote its objects under a fresh prefix and never published a manifest.
try:
    write_table(b, "orders/data/v2-", 100, fail_after=50)
except Failure:
    pass

# The manifest still names version 1's files, so version 1 is still what the table is.
write_table(b, "orders/data/v1-", 3)
b.put("orders/manifest.json", json.dumps(sorted(b.list("orders/data/v1-"))).encode())


def read_table(bucket):
    files = json.loads(bucket.get("orders/manifest.json"))
    return [bucket.get(k) for k in files]


print("objects under the table prefix:", len(b.list("orders/data/")))
print("rows the table actually has   :", len(read_table(b)))

objects under the table prefix: 53
rows the table actually has   : 3


In [10]:
# Publishing the successful version is one put, and there is no moment at which half of it
# is visible.
write_table(b, "orders/data/v3-", 4)
b.put("orders/manifest.json", json.dumps(sorted(b.list("orders/data/v3-"))).encode())
print("rows the table now has:", len(read_table(b)))
print("the 50 orphans are still in the bucket:", len(b.list("orders/data/v2-")))

rows the table now has: 4
the 50 orphans are still in the bucket: 50


## Conclusion

Three properties of a flat key space, measured rather than asserted:

* a prefix "rename" costs $2N + 1$ requests and has no atomic instant, so a commit protocol
  built on directory rename is both slow and unsafe here;
* a prefix listing is a paginated scan of the key index, so a table of many small files is
  expensive to *plan* before it is expensive to read;
* single-object atomicity does not compose, so a failed job leaves a plausible-looking wrong
  table.

The last two cells are the shape of the answer the rest of the chapter develops. Defining the
table by a small piece of metadata, rather than by everything under a prefix, turns the commit
into a single atomic write and leaves the failed job's output as garbage instead of as data.
That is the Iceberg catalog pointer of §6.1 and the Delta commit file of §6.2 — and the orphans
left behind in the last cell are why §6.1's maintenance box has an orphan-file cleanup in it.